# compact

> Controlled conversation compaction: a structured summary with an explicit retain-list, then a
> fresh chat on the same engine seeded with it. Retention is enforced by schema, not hoped for.

In [ ]:
#| default_exp compact

In [ ]:
#| hide
from nbdev.showdoc import *

litert holds the KV cache, so history can't be edited in place -- compaction means: summarize the
old conversation, build a new `Chat` on the same engine, and seed the summary through its system
prompt. Two properties make this *controlled*:

- **the retain-list becomes a schema.** Each item in `retain` becomes a required field of a dynamic
  dataclass passed to `chat.structured()`, so the model must fill every one -- 'information to
  retain' is a contract, not a suggestion. The summary runs in a throwaway conversation on the same
  engine, leaving the live chat intact until the swap.
- **the executor namespace survives untouched.** Variables, fetched pages, kosha indexes all live in
  the `Executor`, not the context; only the *conversation* is compressed.

In [ ]:
#| export
import re
from dataclasses import make_dataclass
from fastcore.utils import store_attr

In [ ]:
#| export
def _msg_text(m):
    try:
        from rishi.core import resp_text
        return resp_text(m)
    except Exception:
        if isinstance(m, dict):
            for k in ('contents','content','text'):
                if k in m: return str(m[k])
        return str(m)

def transcript(hist, last:int=None) -> str:
    'Render history message dicts as `role: text` lines (only the final `last` messages if given).'
    msgs = hist[-last:] if last else hist
    return '\n'.join(f"{m.get('role','?') if isinstance(m, dict) else '?'}: {_msg_text(m)}" for m in msgs)

In [ ]:
#| export
def slug(s:str) -> str:
    'A retain instruction as a valid field name.'
    out = re.sub(r'\W+', '_', s.strip().lower()).strip('_')[:40]
    return out if out and not out[0].isdigit() else f'i_{out}'

def retain_schema(retain, name='Retained'):
    'Dataclass with a `summary` field plus one required string field per retain instruction.'
    cls = make_dataclass(name, [('summary', str)] + [(slug(r), str) for r in retain])
    cls.__doc__ = ('Compressed conversation state. summary: what happened and where the task stands.\n'
                   + '\n'.join(f'{slug(r)}: {r}' for r in retain))
    return cls

In [ ]:
#| export
def seed_block(summ, retain) -> str:
    'Render a filled schema instance as the `<conversation-summary>` block for the new system prompt.'
    lines = [getattr(summ, 'summary', str(summ))]
    items = [f'- {r}: {getattr(summ, slug(r), "")}' for r in retain]
    if items: lines += ['', 'Retained information:', *items]
    return '<conversation-summary>\n' + '\n'.join(lines) + '\n</conversation-summary>'

In [ ]:
#| export
def needs_compact(chat, at:float=0.85) -> bool:
    'True when the chat tracks a context limit and is at least `at` full.'
    return bool(getattr(chat, 'ctx_limit', None)) and chat.pct_full >= at

In [ ]:
#| export
def compact(chat,
            retain=(),      # information-to-retain instructions; each becomes a required schema field
            keep_last:int=4,# recent messages carried over verbatim inside the seed
            sp:str=None,    # system prompt for the new chat (default: the old chat's)
            mk_chat=None,   # chat factory (default rishi.core.Chat); receives the kwargs below
            **chat_kw):
    '''Summarize `chat` with a structured retain-list and return a fresh chat on the same engine,
    seeded via its system prompt. The old chat is returned to the caller untouched -- do not `close()`
    it if it owns an engine the new chat shares.'''
    if mk_chat is None: from rishi.core import Chat as mk_chat
    summ = chat.structured('Summarize the conversation below for a fresh assistant taking over '
                           'mid-task. Fill every field; be specific and factual.\n\n' + transcript(chat.hist),
                           retain_schema(retain))
    seed = seed_block(summ, retain)
    if keep_last and chat.hist: seed += f'\n\nMost recent turns:\n{transcript(chat.hist, last=keep_last)}'
    base_sp = sp if sp is not None else getattr(chat, 'sp', '') or ''
    kw = dict(engine=getattr(chat, 'engine', None), sp=(base_sp + '\n\n' + seed).strip(),
              tools=getattr(chat, 'tools', None), approve=getattr(chat, 'approve', None),
              ctx_limit=getattr(chat, 'ctx_limit', None))
    kw.update(chat_kw)
    return mk_chat(**kw)

In [ ]:
s = retain_schema(['Open task list', 'Files touched so far'])
assert [f for f in s.__dataclass_fields__] == ['summary', 'open_task_list', 'files_touched_so_far']
inst = s(summary='did things', open_task_list='1. finish', files_touched_so_far='a.py')
blk = seed_block(inst, ['Open task list', 'Files touched so far'])
assert 'did things' in blk and '- Open task list: 1. finish' in blk and blk.startswith('<conversation-summary>')
assert slug('2 fast 2 furious').startswith('i_')

In [ ]:
class _FakeChat:
    'Quacks like rishi.Chat for the pieces compact() touches.'
    def __init__(self, **kw): self.kw, self.hist, self.sp = kw, [], kw.get('sp','')
    engine, tools, approve, ctx_limit, pct_full = 'ENG', None, None, 100, 0.9
    def structured(self, prompt, schema):
        assert 'user: hello' in prompt          # transcript made it into the summary prompt
        flds = {f: f'v_{f}' for f in schema.__dataclass_fields__}
        return schema(**flds)

old = _FakeChat(); old.hist = [{'role':'user','contents':'hello'}, {'role':'model','contents':'hi'}]
new = compact(old, retain=['Open task list'], mk_chat=_FakeChat)
assert new.kw['engine'] == 'ENG' and new.kw['ctx_limit'] == 100
assert 'v_summary' in new.kw['sp'] and '- Open task list: v_open_task_list' in new.kw['sp']
assert 'Most recent turns:' in new.kw['sp']
assert needs_compact(old)
new.ctx_limit = None
assert not needs_compact(new)   # no ctx_limit tracked -> never auto-compacts


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()